# Non-Financial Rewards, Employee Engagement, Culture, and Innovation Survey Responses from Likert-Scale Measurements Exploration with `mlcroissant`
This notebook walks through the loading and analysis of the "Non-Financial Rewards, Employee Engagement, Culture, and Innovation Survey Responses from Likert-Scale Measurements" dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and describes survey responses assessing non-financial rewards, engagement, innovation, and cultural dimensions among employees using Likert scales.

In [ ]:
# Ensure `mlcroissant` is installed (uncomment if running interactively)
!pip install mlcroissant --quiet


## 1. Data Loading
Load the dataset metadata and instantiate the dataset with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.arnw-0049/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Print metadata overview
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
List available record sets (tables), their fields, and IDs using their `@id`.
Entities are always referenced by their `@id` for clarity and reproducibility.


In [ ]:
# Discover record sets in the dataset
record_sets = list(dataset.list_record_sets())
print(f"Available record sets (by @id):\n")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '')}")


# Show sample field information for each record set
for rs in record_sets:
    print(f"\nRecord set {rs['@id']} fields:")
    for field in rs.get('field', []):
        field_id = field['@id']
        fname = field.get('name', '')
        ftype = field.get('dataType', '')
        print(f"  - {field_id} (name: {fname}, type: {ftype})")

## 3. Data Extraction
We next load the content (records) from one or more record sets as DataFrames, referencing the record set and field `@id`s found above. All accesses are via the entity `@id` values for consistency.

**Note:** The following example loads all available record sets, but you can pick those relevant to your analysis.

In [ ]:
dataframes = {}
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Loading records for record set {rs_id} ...")
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    if not dataframes[rs_id].empty:
        print(f"First columns for {rs_id}: {list(dataframes[rs_id].columns)}\n")
        display(dataframes[rs_id].head(3))
    else:
        print(f"Record set {rs_id} is empty.\n")

## 4. Exploratory Data Analysis (EDA)
Apply some typical EDA steps: filtering, normalization, and grouping, using field `@id`s for reference.

Adjust field `@id` values and processing as appropriate for your use-case based on the actual data overview above.

In [ ]:
# For demonstration, pick the first non-empty record set.
import numpy as np

selected_rs_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        selected_rs_id = rs_id
        break
if selected_rs_id is None:
    raise RuntimeError("No data available to analyze.")

print(f"Exploring record set: {selected_rs_id}")

# Infer a numeric field by datatype or column name: Example uses any Likert scale column.
df = dataframes[selected_rs_id]
print("Columns:", df.columns.tolist())

# Guess a likely numeric field (starts with 'Q1', etc. or contains 'vigor', 'recognition', etc.)
numeric_candidates = [c for c in df.columns if df[c].dtype.kind in 'iuf' or any(x in c.lower() for x in ['vigor', 'recognition', 'reward', 'engagement', 'likert', 'score', 'q', 'autonomy', 'innovation'])]
if not numeric_candidates:
    numeric_field = df.select_dtypes(include=[np.number]).columns[0]
else:
    numeric_field = numeric_candidates[0]
print(f"Using numeric field (by @id or col name): {numeric_field}")

# Filtering for high values (Likert > 3 as an example)
threshold = 3
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold} (records shown: {min(5, len(filtered_df))}):")
display(filtered_df.head(5))

# Normalize column (z-score)
norm_col = f"{numeric_field}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' for filtered records:")
display(filtered_df[[numeric_field, norm_col]].head())

# Group by a categorical attribute (e.g. 'gender', 'role', etc.) if available
possible_group_by = [c for c in df.columns if any(x in c.lower() for x in ['gender', 'role', 'group', 'education', 'department'])]
if possible_group_by:
    group_field = possible_group_by[0]
    print(f"\nGrouping results by '{group_field}':")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame().rename(columns={numeric_field: f'mean_{numeric_field}'})
    display(grouped_df.head())
else:
    print("No suitable group-by fields found.")

## 5. Visualization
Visualize filtered or grouped results to explore Likert scale response distributions and relationships between key attributes.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of Likert numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field].dropna(), bins=5, kde=True)
plt.xlabel(numeric_field)
plt.title(f'Distribution of {numeric_field}')
plt.show()

# Boxplot by group (if group_field exists)
if possible_group_by:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f'{numeric_field} distribution by {group_field}')
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 survey dataset describing workplace engagement, rewards, culture, and innovation, explored its record sets and fields, filtered, normalized, and visualized Likert responses, and grouped results by key demographics.

This approach can be extended to more complex analyses or models as needed, with full traceability to schema entities via their Croissant `@id`s.